# 03 — Detección YOLOv8 + tracking DeepSORT (Fase 0)

**Objetivo:** IDs estables por objeto en secuencia corta; exportar `tracks.jsonl` y video anotado.


## Prerrequisitos

Salida de **02** (`manifest.json` + clips).


## 1. Setup


In [6]:
from __future__ import annotations

from pathlib import Path

import cv2
import numpy as np
from deep_sort_realtime.deepsort_tracker import DeepSort
from ultralytics import YOLO

from _common.io import (
    append_jsonl,
    ensure_scripts_on_path,
    read_json,
    setup_logging,
    stage_output_dir,
)
from loguru import logger

ensure_scripts_on_path()
setup_logging()


## 2. Configuration


In [7]:
SEGMENTS_DIR = stage_output_dir("02_segments")
OUT_DIR = stage_output_dir("03_track")
TRACKS_PATH = OUT_DIR / "tracks.jsonl"

CLIP_ID = "clip_0000"
YOLO_MODEL = "yolov8n.pt"
CONF_THRESHOLD = 0.35
# COCO: person=0; truck=7, car=2 como proxy de montacargas
TARGET_CLASS_IDS = {0, 2, 7}
CLASS_NAMES = {0: "person", 2: "car", 7: "truck"}


## 3. Cargar clip y modelos


In [8]:
manifest = read_json(SEGMENTS_DIR / "manifest.json")
clip_entry = next((c for c in manifest["clips"] if c["clip_id"] == CLIP_ID), manifest["clips"][0])
from _common.io import repo_root

clip_dir = repo_root() / clip_entry["path"]

frame_paths = sorted(clip_dir.glob("frame_*.jpg"))
if not frame_paths:
    raise FileNotFoundError(f"Sin frames en {clip_dir}")

model = YOLO(YOLO_MODEL)
tracker = DeepSort(max_age=30, n_init=3)
logger.info("Procesando {} frames de {}", len(frame_paths), clip_entry["clip_id"])


20:20:34 | INFO | Procesando 58 frames de clip_0000


## 4. Detección + tracking por frame


In [9]:
if TRACKS_PATH.exists():
    TRACKS_PATH.unlink()

h, w = cv2.imread(str(frame_paths[0])).shape[:2]
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(OUT_DIR / "annotated.mp4"), fourcc, 10.0, (w, h))

for frame_idx, fp in enumerate(frame_paths):
    frame = cv2.imread(str(fp))
    results = model(frame, verbose=False)[0]
    detections = []
    for box in results.boxes:
        cls_id = int(box.cls.item())
        if cls_id not in TARGET_CLASS_IDS:
            continue
        conf = float(box.conf.item())
        if conf < CONF_THRESHOLD:
            continue
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        detections.append(([x1, y1, x2 - x1, y2 - y1], conf, CLASS_NAMES.get(cls_id, str(cls_id))))

    tracks = tracker.update_tracks(detections, frame=frame)
    for tr in tracks:
        if not tr.is_confirmed():
            continue
        l, t, bw, bh = map(int, tr.to_ltrb())
        tid = tr.track_id
        label = tr.get_det_class() if hasattr(tr, "get_det_class") else "obj"
        append_jsonl(
            TRACKS_PATH,
            {
                "frame_idx": frame_idx,
                "track_id": int(tid),
                "class": str(label),
                "bbox": [l, t, l + bw, t + bh],
                "confidence": float(tr.det_conf) if tr.det_conf else None,
            },
        )
        cv2.rectangle(frame, (l, t), (l + bw, t + bh), (0, 255, 0), 2)
        cv2.putText(frame, f"{tid}:{label}", (l, max(t - 5, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    writer.write(frame)

writer.release()
track_count = sum(1 for _ in open(TRACKS_PATH))
logger.info("Registros en tracks.jsonl: {}", track_count)


20:20:51 | INFO | Registros en tracks.jsonl: 224


## 5. Validación


In [10]:
from _common.io import read_jsonl
rows = read_jsonl(TRACKS_PATH)
assert len(rows) > 0, "tracks.jsonl vacío"
ids = {r["track_id"] for r in rows}
print(f"OK — {len(rows)} detecciones, {len(ids)} track IDs únicos")


OK — 224 detecciones, 4 track IDs únicos


## Siguiente paso

**[04_build_event_buffer.ipynb](04_build_event_buffer.ipynb)**
